# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration and processing of the FAIR^2 dataset using the `mlcroissant` library, following Croissant schema best practices.

### Dataset Source
The dataset is described by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using the Croissant schema and `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
List record sets available in the dataset along with their fields and `@id` references.

In [ ]:
# Retrieve all record set @id values and fields
record_sets = [r for r in dataset.record_sets]
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"\nRecord Set Name: {rs.name}\n@id: {rs.id}\nFields:")
        for field in rs.fields:
            print(f"  - Field: {field.name}, @id: {field.id}, dtype: {field.data_type}")

## 3. Data Extraction
Load data from each record set into pandas DataFrames using their `@id` for reference.

In [ ]:
# Create a mapping from record set name to @id for convenience
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for record set {record_set_id} (shape: {df.shape})")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head(3))

# For subsequent analysis, select the first (primary) record set, if present.
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Performing simple EDA: filtering records on a numeric field, normalization, and groupby using field `@id`s __(if available)__.

In [ ]:
if not main_record_set_id or main_record_set_id not in dataframes:
    print("No main record set with data to analyze.")
else:
    df = dataframes[main_record_set_id]
    # List all fields with data type information from metadata
    field_map = {field.id: field for rs in record_sets for field in rs.fields if rs.id == main_record_set_id}
    numeric_fields = [fid for fid, f in field_map.items() if f.data_type in ('Float', 'Integer', 'Number') and fid in df.columns]
    # For demonstration, pick the first available numeric field
    if not numeric_fields:
        print("No numeric fields found for demonstration.")
    else:
        numeric_field_id = numeric_fields[0]
        # Show basic statistics
        print(f"Using numeric field: {numeric_field_id}\nSummary Stats:")
        print(df[numeric_field_id].describe())
        threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a likely categorical field (pick the first 'Text' field in the record set)
        group_fields = [fid for fid, f in field_map.items() if f.data_type == 'Text' and fid in df.columns]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by {group_field_id}...")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            display(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field and grouped statistics. (Requires matplotlib or seaborn)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_record_set_id or main_record_set_id not in dataframes or not numeric_fields:
    print("No data or numeric fields available for visualization.")
else:
    df = dataframes[main_record_set_id]
    num_col = numeric_field_id

    plt.figure(figsize=(7,4))
    sns.histplot(df[num_col].dropna(), kde=True, bins=15, color='steelblue')
    plt.title(f"Distribution of {num_col}")
    plt.xlabel(num_col)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id in df.columns:
        # Bar plot of group means
        group_means = df.groupby(group_field_id)[num_col].mean().sort_values()
        plt.figure(figsize=(8,4))
        sns.barplot(y=group_means.index, x=group_means.values, palette="mako")
        plt.title(f"Mean of {num_col} by {group_field_id}")
        plt.xlabel(f"Mean {num_col}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook used the `mlcroissant` library to load a FAIR dataset following the Croissant schema.
- Explored record sets and fields via their `@id`s; loaded tabular data into pandas.
- Demonstrated basic EDA: filtering, normalization, grouping, and simple visualizations.
- The approach is adaptable to any FAIR Croissant-compliant dataset: update the schema URL and run!

**For further analysis, refer to the Croissant documentation and inspect entities and annotation structure using their persistent `@id` identifiers.**